<a href="https://colab.research.google.com/github/mnsbharadwaj/AI-NLP/blob/master/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**RAG : Retrieval-Augmented Generation**

RAG (Retrieval-Augmented Generation) is a powerful method that combines retrieval-based and generation-based techniques for answering questions or generating text. The core idea behind RAG is to retrieve relevant documents (contexts) and then use them to generate accurate, grounded, and fluent responses.

**Core Components of RAG:**

RAG consists of two major components:


**Retriever (Dense Retriever)** : Retrieves top-k relevant documents using embeddings.

**Generator** (e.g., BART, T5):	Generates final answer based on retrieved documents.


**Retriever (Dense Passage Retriever - DPR)**

  Purpose:
Convert input queries and documents into dense vectors (embeddings) and retrieve the top-k similar documents.

Key Modules:

Context Encoder: Embeds documents.

Question Encoder: Embeds query.

Similarity Search: Typically via FAISS.

In [1]:
!pip install faiss-cpu
!pip install chromadb

In [2]:
from transformers import DPRQuestionEncoder, DPRContextEncoder, DPRQuestionEncoderTokenizer, DPRContextEncoderTokenizer
import torch
import faiss

# Load pretrained encoders and tokenizers
question_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")

context_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")

# Sample corpus
documents = [
    "The capital of France is Paris.",
    "Hugging Face provides transformers for NLP.",
    "The Moon orbits the Earth.",
]

# Encode documents
def encode_documents(docs):
    inputs = context_tokenizer(docs, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        embeddings = context_encoder(**inputs).pooler_output
    return embeddings

doc_embeddings = encode_documents(documents)

# FAISS index
index = faiss.IndexFlatL2(doc_embeddings.size(1))  # dimension
index.add(doc_embeddings.numpy())

# Encode query
query = "What is the capital of France?"
query_inputs = question_tokenizer(query, return_tensors='pt')
with torch.no_grad():
    query_embedding = question_encoder(**query_inputs).pooler_output

# Retrieve top-k (e.g., 2) documents
top_k = 2
distances, indices = index.search(query_embedding.numpy(), top_k)

# Show results
print("Query:", query)
for idx in indices[0]:
    print("Retrieved:", documents[idx])


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This 

Query: What is the capital of France?
Retrieved: The capital of France is Paris.
Retrieved: The Moon orbits the Earth.


## **Generator (e.g., BART, T5)**
Purpose:

Generates an answer conditioned on the retrieved documents.

How It Works:

Takes [query + retrieved_document] as input.

Generates an answer using a seq2seq transformer (e.g., facebook/bart-large or google/flan-t5).

In [3]:
from transformers import BartTokenizer, BartForConditionalGeneration

# Load BART generator
gen_tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')
generator = BartForConditionalGeneration.from_pretrained('facebook/bart-large')

# Combine query and top retrieved document
retrieved_context = documents[indices[0][0]]
input_text = f"question: {query} context: {retrieved_context}"
inputs = gen_tokenizer(input_text, return_tensors='pt', truncation=True, padding=True)

# Generate answer
with torch.no_grad():
    output_ids = generator.generate(**inputs, max_length=50)

print("Generated Answer:", gen_tokenizer.decode(output_ids[0], skip_special_tokens=True))


Generated Answer: question: What is the capital of France? context: The president of France is Paris.


## **Code Generation**

In [6]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("microsoft/codebert-base")  # or "BAAI/bge-code" or "intfloat/e5-base-v2"


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [7]:
def chunk_code(code_str):
    return code_str.split("\n\n")  # naive split by function


In [1]:
!pip install chromadb sentence-transformers transformers
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection(name="code_chunks")

# Replace chunk1, chunk2, ... with your actual code chunks
# Replace [...] with a list of unique ids for your chunks
# Replace [{"filename": "utils.py"}, ...] with appropriate metadata for each chunk
# For example:
# code_chunks = chunk_code(your_code_string)
# chunk_ids = [f"chunk_{i}" for i in range(len(code_chunks))]
# chunk_metadatas = [{"filename": "your_file_name.py"}] * len(code_chunks)

# collection.add(
#     documents=code_chunks,
#     ids=chunk_ids,
#     metadatas=chunk_metadatas
# )

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.6/201.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.9 MB/s eta